---
title: "Trotterized Hubbard Model"
title-block-banner: true
abstract: |
  This article evaluates novel approaches to do
  some really important things.
---

todo: Check neel state set up

Generic resource estimation overestimates Trotter cost for structured Hamiltonians like Hubbard by X. We show this empirically using spectral norm, commutator bounds, and state-specific fidelity on a 2×2 model. We then demonstrate how Hubbard's bipartite and commutative structure can be exploited to reduce the resource estimate by factor Y

# Motivation for Trotterization

Why QSVT with QROM vs Trooterization

Hubbard is local and structured
QSVT/QROM is especially powerful when the Hamiltonian is given as a large irregular table: But the Hubbard model on a regular lattice is not an arbitrary table. It has simple repeated structure:

- nearest-neighbor hopping,
- osite interaction,
- mostly uniform coefficients,
- sparse local geometry.

In [6]:
# | code-fold: true
# | code-summary: "Import libraries"
import logging

import matplotlib.pyplot as plt
import numpy as np
import pennylane as qp
import pennylane.estimator as qre
from pennylane.resource import SpectralNormError
import tqdm

logging.basicConfig(level=logging.INFO)

# Setup {#sec-setup}

We create a $2 \times 2$ rectangle, with 4 sites, and 8 qubits. The notion is


| Qubit | Spin orbital |
|-------|--------------|
| 0     | site 0 ↑     |
| 1     | site 0 ↓     |
| 2     | site 1 ↑     |
| 3     | site 1 ↓     |
| 4     | site 2 ↑     |
| 5     | site 2 ↓     |
| 6     | site 3 ↑     |
| 7     | site 3 ↓     |



In [5]:
t = 1.0
U = 4.0
time = 1  # The time of evolution (t in exp(-iHt))

# number of Trotter steps, proposally not good to show the power of higher Trotter order
n_steps = int(time * 5)

n_cells = [2, 2, 1]
orders = [1, 2]
H = qp.spin.fermi_hubbard(
    lattice="cubic",
    n_cells=n_cells,
    hopping=t,
    coulomb=U,
    boundary_condition=False,
    mapping="jordan_wigner",
)
n_qubits = H.num_wires

This link is interesting https://fermi-hubbard-commutators.readthedocs.io/en/latest/commutator_bounds.html

# Naive Trotter error bound

How badly do standard Trotter error bounds overestimate cost for specific observables in Hubbard, and what's the practical implication for resource estimates?"
The spectral norm $\lVert {U_{exact} - U_{trotter}}\rVert_2$ measures the maximum possible deviation over all possible input states in the entire Hilbert space.

However in common application, these relevant states (e.g, low-energy excited states with specific symmetries) is in a much smaller space. Therefore it is not the best interest to tune the Trotter using the Spectral Norm.

The next natural question is for the Hubbard model, the most relavant states are? In this example we use half-filled Neel state ($\ket{0101...01}$), which is standard for studying Mott insulators.

Haar-random states are common in complexity proofs because they are uniformly distributed across the full Hilbert space, making them worst-case inputs. However, they have no physical relevance to Hubbard physics.

Compute grouping here  https://docs.pennylane.ai/en/stable/code/api/pennylane.ops.op_math.LinearCombination.html#pennylane.ops.op_math.LinearCombination.compute_grouping

In [39]:
# H.compute_grouping()  # compute the qubit-wise commuting groups!

# resources_exec = qre.estimate(executable_circuit)(grouped_hamiltonian, num_steps, order)

# resources_with_grouping = qre.estimate(
#     qre.TrotterPauli(kitaev_H_with_grouping, num_steps, order)
# )

# res = qre.estimate(circuit)(kitaev_H_with_grouping, num_steps, order)

In [ ]:
exact_op = qp.exp(H, -1j * time)


## Spectral Norm {#secsec-spectral-norm}

Exact calculation, with such a small system, the Hamiltonian matrix is tractable ad diagonalizable. It stuggles at 16 qubits, which would result in $2^{16} \times 2^{16}$ matrix. However, due to the locality nature of Hubbard model, we can save a lot of memory using the sparse matrix

In [ ]:
exact_op = qp.exp(H, -1j * time)
spectral_error = {}

for order in tqdm.tqdm(orders):
    approx_op = qp.TrotterProduct(H, n=n_steps, time=time, order=order)
    error = SpectralNormError.get_error(exact_op, approx_op)
    spectral_error[order] = error

print(
    "\n".join(
        [f"Order: {k}, Spectral error: {v:5f}" for k, v in spectral_error.items()]
    )
)

  0%|          | 0/2 [00:00<?, ?it/s]

## Childs Method

Since Calculating spectral norm is expensive because it requires diagonalizing the operators.

In [41]:
for order in tqdm.tqdm(orders):
    op = qp.TrotterProduct(H, n=n_steps, time=time, order=order)

    one_norm_error_bound = op.error(method="one-norm-bound")
    commutator_error_bound = op.error(method="commutator-bound")

    print(f"Order {order}")
    print(f"One-norm bound: {one_norm_error_bound}")
    print(f"Commutator bound: {commutator_error_bound}")

 50%|█████     | 1/2 [00:01<00:01,  1.42s/it]

Order 1
One-norm bound: SpectralNormError(1296.0)
Commutator bound: SpectralNormError(324.0)


100%|██████████| 2/2 [02:32<00:00, 76.01s/it]

Order 2
One-norm bound: SpectralNormError(34992.0)
Commutator bound: SpectralNormError(2232.0)


This seems counter intuitive but makes sense because higher order's error should scale with "this math", but the 
Order 2 is better algorithmically but has worse worst-case combinatorial bound.

One-norm bound is faster because it just adds every possible error

Commutator bound is tighter here it finds every pair $i, j$ such that [Hᵢ, Hⱼ] = 0. For the Hubbard model, all the interaction terms V = Σ U nᵢ↑nᵢ↓ commute with each other, so this pruning is substantial.
Why it runs out of memory at high orders — the combinatorial explosion:
For an order-p Trotter formula, the commutator bound requires computing nested commutators to depth p+1. With N Hamiltonian terms:
Trotter orderCommutator depth needed# evaluations1depth 2~N²2depth 3~N³4depth 5~N⁵

So we spent a lot of time to calculate these bounds, but end up not gaining anything very informative about how to set the parameters. In larger system, the calculation of SpectralNorm error we did in @secsec-spectral-norm is imposible. 

Physicists only care for limited interesting states, and they live in a small manifold in the Hilbert space. Here we propose using state-aware error estimation

# Physics state-awared error bound estimation

- What are the super set of Neel state? I choose half filled state
- How does state-dependent error suppression affect the optimal Trotter order?
    - Perhaps with this we see 1-order is already very good
    - 2nd order meh
    - 4th order unncesscary
    - In the end "Do physically relevant states reduce the practical advantage of higher-order Trotter formulas in Hubbard simulation?"

## State initialization

Neel state ($\ket{0101...01}$) is the standard state for studying ferromagnetics problem. Using the notation in @sec-setup, we deduce that

In [42]:
def prepare_neel_state(wires):
    """Néel state: alternating up/down on bipartite lattice"""
    for site in range(wires // 2):
        if site % 2 == 0:
            qp.PauliX(wires=2 * site)
        else:
            qp.PauliX(wires=2 * site + 1)

In [48]:
dev = qp.device("default.qubit")


@qp.qnode(dev)
def exact_circ(H, t, wires):
    """
    Simluate exact evolution
    """
    prepare_neel_state(wires)
    qp.exp(-1j * t * H)
    return qp.state()


@qp.qnode(dev)
def trotter_circ(H, num_steps, t, wires, order=1):
    prepare_neel_state(wires)
    qp.TrotterProduct(H, n=num_steps, time=t, order=order)
    return qp.state()

In [49]:
resource_est = {}
for order in orders:
    resources_exec = qre.estimate(trotter_circ)(H, n_steps, time, n_qubits, order=order)
    resource_est[order] = resources_exec.gate_counts

In [ ]:
resource_est

In [ ]:
exact_state = exact_circ(H, time, n_qubits)
error_dicts = {}
state_fidelities = {}

for order in orders:
    trotter_state = trotter_circ(H, n_steps, time, n_qubits, order=order)
    state_fidelities[order] = qp.math.fidelity_statevector(exact_state, trotter_state)
    # error_dicts[order] = qp.resource.algo_error(trotter_circ)(
    #     H, n_steps, time, n_qubits, order=order
    # )

In [ ]:
state_fidelities

In [ ]:
error_array = spectral_error.values()
gates = [resource_est[order]["T"] for order in orders]
plt.plot(gates, error_array)
plt.ylabel("Spectral Error")
plt.xlabel("T gates")
plt.show()

So we need to have >70000 T gates to make a simulation for `{python} time` seconds on a modest-sized Hubbard simulation. Can we do better?

## Hamiltonian properties

We have two types of interaction in Hubbard: Hopping $T$ and 

Hubbard physics is about local correlations, hopping, double occupancy penalties, and half-filling.
I expect a lot of clever commutator hack, and Hubbard model exploitation


State-dependent commutator bound

The leading Trotter error operator is

$δ2/2j<k∑[Hj,Hk]$

Instead of ∥E_2∥, consider

∥E_2 ∣ψ_0⟩∥

Then $ϵ_ψ​≤δ^2/2 ​⟨ψ0​|​j<k∑​[Hj​,Hk​]​†(l<m∑​[Hl​,Hm​])​ψ0​⟩$


This is automatically tighter.

Property 1: The interaction terms are fully diagonal — [V, V] = 0 exactly
All terms in your Hamiltonian that do not contain X or Y are diagonal in the computational basis:
4·I, -Z(i), Z(i)⊗Z(j). These are all Z-strings, which mutually commute. The consequence is zero intra-group Trotter error for V — e^{-iVt} is implemented exactly with no decomposition penalty.

Property 2: Bipartite sublattice decomposition — [T_H, T_H] = 0 and [T_V, T_V] = 0
Split the hopping into horizontal bonds (qubit span = 2: Y(0)Z(1)Y(2), X(0)Z(1)X(2), ...) and vertical bonds (span = 4: Y(0)Z(1)Z(2)Z(3)Y(4), ...).
Two hopping terms commute iff the number of positions where they anticommute is even. For two horizontal hops on qubits {0,1,2} and {1,2,3}: they anticommute at positions 1 and 2 (Z vs X, and X vs Z) → two anticommuting positions → they commute. You can verify this holds for all pairs within T_H and all pairs within T_V.
This means: H = T_H + T_V + V, where every term within each group commutes. The Trotter error comes only from the three cross-commutators: [T_H, T_V], [T_H, V], [T_V, V] — not from the N(N-1)/2 ≈ 406 pairs you'd count treating each Pauli term independently. This is the group-commuting bound.

Property 3: Conservation laws restrict the relevant Hilbert space by 7×
The Hubbard Hamiltonian conserves both total particle number N and total spin Sz. Each term in your Hamiltonian also individually conserves these (X and Y terms always flip a pair of qubits with opposite occupancies). The Néel state has N=4, Sz=0.
The dimension of the N=4, Sz=0 sector is C(4,2) × C(4,2) = 36, vs the full 256. Any spectral norm computation restricted to this 36×36 block bounds the error for any state that starts (and stays) in this sector — which is exactly your simulation.

Property 4: The Néel state has two exact cancellations at leading order
This is the sharpest property. For your specific initial state (qubits 0, 3, 4, 7 occupied):
V|ψ₀⟩ = 0. The Néel state has zero double occupancy. Each site has exactly one electron. With U=4, computing V = 4I + Σᵢ(-Z(2i) - Z(2i+1) + Z(2i)Z(2i+1)), every site contributes -1 - 1 + 1 - 1 - 1 + 1 - 1 ... giving v₀ = 4 - 4 = 0 exactly.
T_V|ψ₀⟩ = 0. Vertical hops connect site 0↑ (q₀=1) to site 2↑ (q₄=1), and site 0↓ (q₁=0) to site 2↓ (q₅=0). Both pairs are (1,1) or (0,0) — Pauli exclusion blocks both. All four vertical hop pairs are in the same (1,1) or (0,0) configuration.
Therefore for the first-order Trotter error terms involving the Néel state:

[T_H, T_V]|ψ₀⟩ = T_H(T_V|ψ₀⟩) - T_V(T_H|ψ₀⟩) = 0 - T_V(T_H|ψ₀⟩) → only second-order contribution
[T_H, V]|ψ₀⟩ = T_H(V|ψ₀⟩) - V(T_H|ψ₀⟩) = 0 - V(T_H|ψ₀⟩) → bounded by U × ‖T_H|ψ₀⟩‖
[T_V, V]|ψ₀⟩ = T_V(V|ψ₀⟩) - V(T_V|ψ₀⟩) = 0 - 0 = exactly zero

Two of three leading error terms vanish identically at the initial state. This is why fidelity error << spectral norm for this state.

# Business output

If we increase the order based on the Spectral norm estimation, then the number of gates vs accuracy looks like this

But if we increase the order based on the state fidelity, then the number of gates vs accuracy looks like this


# Goal
- Practical resource estimation for early-to-intermediate fault-tolerant quantum simulation of strongly correlated materials. Choose 3D Hubbard model using PennyLane's Labs estimator
- Give systematic, empirical comparison of worst-case vs. observable-specific Trotter error bounds
- Able to distinguish asymptotic theory from realistic hardware constraints;
- Capable of producing technically honest estimates that are useful for strategic decision-making inside an FTQC organization.

https://pennylane.ai/demos/tutorial_estimator_hamiltonian_simulation_gqsp: Why does this increase with time